# v125 — 検出集合固定の cross-proposal 輪郭投票 eval (fold-0, Colab)

consensus 線は fold-0 で閉鎖 (+0.0030 < +0.005)。別機構 P2:
prod 検出集合を leader として**固定**し、crop/yolo26 提案を
mask-IoU>=0.5 で対応付け、refined mask の等重み pixel 多数決
(>=0.5) で輪郭のみ改善。新規個体は出さない。

| arm | mask 合成 |
|---|---|
| `prod` | leader refined のみ (対照、sanity ≈0.444137) |
| `vote_eq05` | leader+voter refined の多数決 (>=0.5 固定) |

**判定** (事前固定・開発側):
- vote_eq05 Δ@0.45 ≥ +0.005 → `vote_positive_dev` (確認 fold へ検討)
- 未満 → `vote_negative` で P2 線も閉鎖
- sanity: prod arm ≈ fold-0 prod 確定値 0.444137 ±0.001
- fold-0 は v121c 判定済み: 陽性でも確認は別 fold が必要

**Provenance pin:** `V73_FOLD0_PROVENANCE_SHA256` (v121c と同一)。
yolo det は manifest binding、crop/yolo26 det は v121a/v121b 学習
manifest 記録の weights_sha256 を trusted-pin として受理。
Drive `solar_filament_v35/kaggle.json` で kaggle 認証 (shingo257)。
eval 終了時に Drive `solar_filament_v125/work/latest/` へ sha256 検証付きミラー。


In [ ]:
import os, re, json, subprocess, sys, hashlib, shutil, time
from pathlib import Path
# Reviewed provenance pin is supplied externally — env var, Colab secret,
# Kaggle secret — or declared interactively. Never derived from artifacts.
EXPECTED_PROVENANCE_SHA = os.environ.get('V73_FOLD0_PROVENANCE_SHA256', '')
if not EXPECTED_PROVENANCE_SHA:
    try:
        from google.colab import userdata
        EXPECTED_PROVENANCE_SHA = userdata.get('V73_FOLD0_PROVENANCE_SHA256') or ''
    except Exception:
        pass
if not EXPECTED_PROVENANCE_SHA:
    try:
        from kaggle_secrets import UserSecretsClient
        EXPECTED_PROVENANCE_SHA = UserSecretsClient().get_secret('V73_FOLD0_PROVENANCE_SHA256')
    except Exception:
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    try:
        EXPECTED_PROVENANCE_SHA = input('V73_FOLD0_PROVENANCE_SHA256 (fold-0 reviewed manifest hash): ').strip()
    except (EOFError, OSError):
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    raise RuntimeError('Set V73_FOLD0_PROVENANCE_SHA256 after independent provenance review (§2.99: da3d53ab...)')

# [0] Drive mount — primary data source (same pattern as all past colab runs)
from google.colab import drive
drive.mount('/content/drive')
V30 = Path('/content/drive/MyDrive/solar_filament_v30')
COMP = V30 / 'competition_data/MAGFiLO_1.0_Kaggle_2026'
inputs = {
    'train_json': COMP / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    'test_dir': COMP / 'test/test_images',
    'preproc_dir': V30 / 'preprocessed_v30_q96',
    'folds_csv': V30 / 'artifacts/grouped_folds.csv',
    'baseline_ckpt': V30 / 'artifacts/mediseg_edgeattnet_refiner_fold0_resume.pt',
    'yolo_ckpt': V30 / 'runs/yolo_fold_0/weights/best.pt',
    'crop_ckpt': V30 / 'runs/v121_croptrain_f0_honest/yolo_croptrain_fold0/weights/best.pt',
    'yolo26_ckpt': V30 / 'runs/v121_yolo26_f0/yolo_yolo26m_fold0/weights/best.pt',
    'v121a_manifest': V30 / 'artifacts/v121a_croptrain_f0_colab_manifest.json',
    'v121b_manifest': V30 / 'artifacts/v121b_yolo26_f0_colab_manifest.json',
    'kaggle.json': Path('/content/drive/MyDrive/solar_filament_v35/kaggle.json'),
}
missing = {k: str(v) for k, v in inputs.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f'Drive inputs missing: {missing}')
train_json, test_dir, preproc_dir = (inputs['train_json'], inputs['test_dir'],
                                     inputs['preproc_dir'])
folds_csv, baseline_ckpt, yolo_ckpt = (inputs['folds_csv'], inputs['baseline_ckpt'],
                                      inputs['yolo_ckpt'])
crop_ckpt, yolo26_ckpt = inputs['crop_ckpt'], inputs['yolo26_ckpt']
# §2.99 — crop/yolo26 fold-0 重みの trusted-pin は各学習 manifest が正本
# (学習時に記録された weights_sha256 を読み取り、eval cell で byte 照合)
_v121a_man = json.loads(inputs['v121a_manifest'].read_text(encoding='utf-8'))
_v121b_man = json.loads(inputs['v121b_manifest'].read_text(encoding='utf-8'))
TRUSTED_SHA256 = {
    str(crop_ckpt): _v121a_man['datasets']['croptrain']['weights_sha256'],
    str(yolo26_ckpt): _v121b_man['results']['yolo26m']['weights_sha256'],
}
print('trusted pins from training manifests:',
      {Path(k).parent.parent.parent.name: v[:12] for k, v in TRUSTED_SHA256.items()})
if not any(test_dir.iterdir()):
    raise RuntimeError('test_images dir is empty')
EXPECTED_FOLDS_SHA = 'febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e'
actual = hashlib.sha256(folds_csv.read_bytes()).hexdigest()
if actual != EXPECTED_FOLDS_SHA:
    raise RuntimeError(f'fold CSV sha mismatch: {actual} (Drive artifacts/grouped_folds.csv)')
print('folds_csv sha256 OK (true experimental split)')

# [1] Kaggle auth + code/manifest download
os.makedirs('/root/.kaggle', exist_ok=True)
cred = inputs['kaggle.json'].read_bytes()
print('auth: Drive kaggle.json')
try:
    _u = json.loads(cred)['username']
except Exception:
    _u = '?'
print('kaggle user:', _u)
Path('/root/.kaggle/kaggle.json').write_bytes(cred)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'kaggle', 'ultralytics', 'pycocotools'], check=True)

WORK = Path('/content/work'); IN = WORK / 'input'; IN.mkdir(parents=True, exist_ok=True)
def _dl(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('download failed rc=%s: %s\n%s\n%s' % (
            r.returncode, cmd, r.stdout[-800:], r.stderr[-800:]))
def _sha(p):
    d = hashlib.sha256()
    with open(p, 'rb') as h:
        for b in iter(lambda: h.read(1 << 20), b''):
            d.update(b)
    return d.hexdigest()

_dl(['kaggle', 'datasets', 'download', 'shingo257/solar-v73-code',
     '-p', str(IN), '--unzip', '-q'])

code_dir = next(p.parent for p in IN.rglob('train_refiner_fold.py'))
sys.path.insert(0, str(code_dir))
from v73_split_provenance import (load_reviewed_manifest, load_fold_mapping,
                                  validate_checkpoint_binding)
# §2.99 — fold-0 reviewed manifest は fold-0 ckpt の独立 pin から
# 決定論的に生成し、自己 sha を外部 pin (V73_FOLD0_PROVENANCE_SHA256)
# で照合する。fold-2 manifest は fold/binding が fold-0 と不一致のため
# 使用しない (checkpoints.yolo/baseline は fold-2 重みの sha)。
FOLD0_YOLO_SHA = 'ab8810916e34a755cc0d7a705dbb7ade60338ae247f8baac02adfe64026c0d35'
FOLD0_REFINER_SHA = 'caa8b27bbdaa888019a599d001b2d4dee844dbfee90186a18f8e0db95ce8b264'
fold0_manifest = {'schema_version': 1, 'fold': 0,
                  'folds_sha256': EXPECTED_FOLDS_SHA,
                  'checkpoints': {'yolo': FOLD0_YOLO_SHA,
                                  'baseline': FOLD0_REFINER_SHA}}
manifest_path = WORK / 'v73_reviewed_provenance_fold0.json'
manifest_path.write_text(json.dumps(fold0_manifest, sort_keys=True,
                                    separators=(',', ':')), encoding='utf-8')
manifest = load_reviewed_manifest(manifest_path, EXPECTED_PROVENANCE_SHA, 0)
if manifest['folds_sha256'] != EXPECTED_FOLDS_SHA:
    raise RuntimeError('reviewed split differs')
coco = json.loads(train_json.read_text())
load_fold_mapping(folds_csv, {i['file_name'] for i in coco['images']}, 0,
                  manifest['folds_sha256'])
validate_checkpoint_binding(yolo_ckpt, 'yolo', manifest)
validate_checkpoint_binding(baseline_ckpt, 'baseline', manifest)

# [2] vote eval variant — embedded copy of repo scratch/run_fold_oof_eval_vote.py (P2・fold-0 実行)
VOTE_SCRIPT = '"""fold N OOF で prod 検出集合を固定した cross-proposal 輪郭投票を採点 (P2/v125)。\n\n`run_fold_oof_eval_consensus.py` (v120/§2.98) からの派生。postprocess は同一:\ndetector fold-N 推論 (imgsz=1280, conf=0.08, iou=0.55, agnostic_nms,\nretina_masks) → refiner で 320px crop の確率マップ → threshold で shrink →\ndrop_contained(0.90) → greedy_disjoint(min_area=20) → 公式 PQ\n(annotator x image group 毎、IoU>0.5 厳密より大きい)。\n\n差分は「何を mask にするか」だけ。検出集合は **prod (yolo detector の\nconf>=--conf 提案) で完全固定** — leader の追加・削除はしない\n(consensus 系の存在フィルタとは別の実験、調査計画 2026-11-10 P2)。\n\narm (事前固定・2 つのみ):\n  prod      : leader の refined mask のみ (対照 = prod pipeline と同一)\n  vote_eq05 : leader refined + voter refined の等重み pixel 多数決\n              (score(p) = 平均 >= 0.5 を保持。N=2 なら intersection、\n              N=3 なら >=2 mask が支持 — voter 2 枚が一致する領域のみ\n              leader 範囲外へ拡張できる。閾値の後付け調整はしない)\n\nvoter 対応付け (事前固定):\n  - 非 `yolo` detector (crop, yolo26) の det-floor 提案のうち、leader の\n    det mask と mask-IoU >= --vote-iou のもの\n  - 各 voter 提案は IoU 最大の leader 1 つにのみ所属 (曖昧 chain で\n    別個体を融合しない)\n  - voter は raw conf を比較しない (detector 間の校正なし・等重み)\n\n`yolo` detector は manifest の `yolo` role で binding、他は\n--trusted-sha256 の byte-pin で受理 (§2.99)。\n\n使い方:\n    python scratch/run_fold_oof_eval_vote.py \\\n        --train-json train/..._train.json --preproc-dir preprocessed_v30_q96 \\\n        --folds-csv grouped_folds.csv --fold 0 \\\n        --provenance-manifest reviewed.json --provenance-sha256 <pin> \\\n        --detector yolo=yolo_fold0_best.pt \\\n        --detector crop=v121a_crop_best.pt --detector yolo26=v121b_best.pt \\\n        --trusted-sha256 <crop_sha> --trusted-sha256 <yolo26_sha> \\\n        --refiner baseline=mediseg_edgeattnet_refiner_fold0_resume.pt \\\n        --out output/v125_eval.json\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\ntry:\n    from scratch.v73_split_provenance import (\n        load_fold_mapping, load_reviewed_manifest, validate_checkpoint_binding,\n        validate_training_corpus_provenance)\nexcept ImportError:\n    from v73_split_provenance import (\n        load_fold_mapping, load_reviewed_manifest, validate_checkpoint_binding,\n        validate_training_corpus_provenance)\ntry:\n    from src.metrics_official import PanopticQuality  # noqa: E402\nexcept ImportError:\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n    from metrics_official import PanopticQuality  # noqa: E402\n\nCROP_SIZE = 320\nCROP_CONTEXT = 1.65\nCONF = 0.26\nMIN_AREA = 20\nDROP_THRESHOLD = 0.90\nEVAL_SIZE = 2048\n\n\ndef load_models():\n    try:\n        from solar_filament.models.mediseg_models import (\n            MediSegEdgeAttUNet, MediSegUNet)\n    except ImportError:\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        from mediseg_models import MediSegEdgeAttUNet, MediSegUNet\n    return MediSegUNet, MediSegEdgeAttUNet\n\n\ndef load_ownership():\n    """canonical apply_ownership を repo / flat dataset 両配置で解決。"""\n    try:\n        from solar_filament.pipeline.ownership import apply_ownership\n    except ImportError:\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        from ownership import apply_ownership\n    return apply_ownership\n\n\ndef to_rle(annotation, height, width, mask_utils):\n    segmentation = annotation["segmentation"]\n    if isinstance(segmentation, list):\n        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))\n    if isinstance(segmentation.get("counts"), list):\n        return mask_utils.frPyObjects(segmentation, height, width)\n    rle = dict(segmentation)\n    if isinstance(rle["counts"], str):\n        rle["counts"] = rle["counts"].encode("utf-8")\n    return rle\n\n\ndef iou_matrix(truth, pred, mask_utils):\n    if not truth or not pred:\n        return np.zeros((len(truth), len(pred)), np.float64)\n    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)),\n                      dtype=np.float64).T\n\n\ndef mask_square_box(mask, context=CROP_CONTEXT, minimum=96):\n    height, width = mask.shape\n    ys, xs = np.where(mask)\n    if not len(xs):\n        return 0, 0, width, height\n    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2\n    side = int(math.ceil(max(xs.max() - xs.min() + 1,\n                             ys.max() - ys.min() + 1) * context))\n    side = min(max(side, minimum), min(height, width))\n    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))\n    x1, y1 = x0 + side, y0 + side\n    if x0 < 0:\n        x1 -= x0\n        x0 = 0\n    if y0 < 0:\n        y1 -= y0\n        y0 = 0\n    if x1 > width:\n        x0 -= x1 - width\n        x1 = width\n    if y1 > height:\n        y0 -= y1 - height\n        y1 = height\n    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)\n\n\n@torch.no_grad()\ndef refine_probability(refiner, image, proposal):\n    x0, y0, x1, y1 = mask_square_box(proposal)\n    crop = cv2.resize(image[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE),\n                      interpolation=cv2.INTER_LINEAR)\n    tensor = (torch.from_numpy(crop.transpose(2, 0, 1)[None].copy())\n              .float().cuda() / 255.0)\n    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()\n    return (cv2.resize(probability, (x1 - x0, y1 - y0),\n                       interpolation=cv2.INTER_LINEAR), (x0, y0, x1, y1))\n\n\ndef result_masks(result):\n    if result.masks is None:\n        return [], np.zeros(0, np.float32)\n    height, width = result.orig_shape\n    output = []\n    for mask in result.masks.data.detach().cpu().numpy():\n        if mask.shape != (height, width):\n            mask = cv2.resize(mask, (width, height),\n                              interpolation=cv2.INTER_LINEAR)\n        output.append(mask >= 0.5)\n    confidences = (result.boxes.conf.detach().cpu().numpy().astype(np.float32)\n                   if result.boxes is not None else np.zeros(0, np.float32))\n    return output, confidences\n\n\ndef sha256_of(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train-json", type=Path, required=True)\n    parser.add_argument("--preproc-dir", type=Path, required=True)\n    parser.add_argument("--test-dir", type=Path, required=True)\n    parser.add_argument("--folds-csv", type=Path, required=True)\n    parser.add_argument("--provenance-manifest", type=Path, required=True,\n                        help="reviewed JSON binding YOLO/legacy weights to fold/split")\n    parser.add_argument("--provenance-sha256", required=True,\n                        help="independently reviewed manifest SHA256; never derive at runtime")\n    parser.add_argument("--fold", type=int, required=True, choices=(0, 1, 2))\n    parser.add_argument("--detector", action="append", required=True,\n                        help="name=ckpt_path (複数指定可。`yolo` は manifest binding、"\n                             "他は --trusted-sha256 pin で受理)")\n    parser.add_argument("--refiner", action="append", required=True,\n                        help="name=ckpt_path (複数指定可、同一提案で採点)")\n    parser.add_argument("--trusted-sha256", action="append", default=[],\n                        help="外部検証済み ckpt の sha256 pin (repeatable)。"\n                             "一致バイトは manifest/metadata binding を代替受理")\n    parser.add_argument("--model", type=str, default="edgeatt",\n                        choices=("unet", "edgeatt"))\n    parser.add_argument("--thresholds", type=str, default="0.35,0.40,0.45,0.50")\n    parser.add_argument("--conf", type=float, default=CONF,\n                        help="prod leader 提案の operating conf")\n    parser.add_argument("--det-floor", type=float, default=0.08,\n                        help="voter detector の提案保持 floor")\n    parser.add_argument("--vote-iou", type=float, default=0.5,\n                        help="voter → leader 対応付けの det-mask IoU 閾値")\n    parser.add_argument("--limit", type=int, default=0)\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n    trusted = set(args.trusted_sha256)\n    manifest = load_reviewed_manifest(\n        args.provenance_manifest, args.provenance_sha256, args.fold)\n    if args.limit < 0:\n        raise ValueError("--limit must be non-negative")\n    thresholds = [float(t) for t in args.thresholds.split(",")]\n    arms = []\n    for spec in args.refiner:\n        name, _, path = spec.partition("=")\n        if (not name or name == "yolo" or name in {n for n, _ in arms}\n                or not path or not Path(path).exists()):\n            raise ValueError(f"--refiner name=path が不正: {spec}")\n        arms.append((name, Path(path)))\n\n    detectors = []\n    for spec in args.detector:\n        name, _, path = spec.partition("=")\n        if (not name or not path or not Path(path).exists()\n                or name in {n for n, _ in detectors}):\n            raise ValueError(f"--detector name=path が不正: {spec}")\n        detectors.append((name, Path(path)))\n    if "yolo" not in {n for n, _ in detectors}:\n        raise ValueError("--detector yolo=... は必須 (prod leader/binding の基準)")\n\n    folds_sha256 = hashlib.sha256(args.folds_csv.read_bytes()).hexdigest()\n    with open(args.train_json, encoding="utf-8") as handle:\n        coco = json.load(handle)\n    images = {i["id"]: i for i in coco["images"]}\n    annotations = defaultdict(list)\n    for ann in coco["annotations"]:\n        annotations[ann["image_id"]].append(ann)\n    file_to_ids = defaultdict(list)\n    for image_id, info in images.items():\n        file_to_ids[info["file_name"]].append(image_id)\n    fold_of_file = load_fold_mapping(\n        args.folds_csv, file_to_ids, args.fold, manifest["folds_sha256"])\n    detector_shas = {}\n    for name, path in detectors:\n        digest = sha256_of(path)\n        if name == "yolo":\n            validate_checkpoint_binding(path, "yolo", manifest)\n        elif digest not in trusted:\n            raise ValueError(\n                f"detector {name} bytes match no reviewed binding and is not "\n                f"in --trusted-sha256: {digest[:16]}")\n        detector_shas[name] = digest\n        print(f"detector {name}: sha256={digest[:16]}", flush=True)\n    yolo_sha256 = detector_shas["yolo"]\n    checked_arms = {}\n    for name, path in arms:\n        checkpoint = torch.load(path, map_location="cpu", weights_only=True)\n        digest = validate_checkpoint_binding(path, name, manifest, checkpoint)\n        provenance = checkpoint.get("init_report", {}).get("corpus_provenance")\n        if str(checkpoint.get("init", "")).startswith("ssl:") or provenance is not None:\n            validate_training_corpus_provenance(provenance, coco, args.test_dir)\n        checked_arms[name] = (checkpoint, digest)\n    # All fold/split and legacy byte bindings pass before any CUDA/model work.\n    oof_files = [n for n in sorted(file_to_ids)\n                 if fold_of_file.get(n) == args.fold]\n    if args.limit:\n        oof_files = oof_files[:args.limit]\n    print({"fold": args.fold, "oof_files": len(oof_files),\n           "arms": [n for n, _ in arms], "folds_sha256": folds_sha256[:16]},\n          flush=True)\n\n    MediSegUNet, MediSegEdgeAttUNet = load_models()\n    apply_ownership = load_ownership()\n    model_cls = MediSegEdgeAttUNet if args.model == "edgeatt" else MediSegUNet\n    from pycocotools import mask as mask_utils\n    from ultralytics import YOLO\n\n    gt_rles = {image_id: [to_rle(a, int(info["height"]), int(info["width"]),\n                                 mask_utils)\n                          for a in annotations[image_id]\n                          if not a.get("iscrowd", 0)]\n               for image_id, info in images.items()}\n\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    refiners = {}\n    for name, path in arms:\n        ckpt, digest = checked_arms.pop(name)\n        model = model_cls(3, 1, False).cuda().eval()\n        model.load_state_dict(ckpt.get("best_model") or ckpt["model"],\n                              strict=True)\n        refiners[name] = {"model": model,\n                          "sha256": digest,\n                          "corpus_provenance": ckpt.get("init_report", {}).get("corpus_provenance"),\n                          "best_dice": ckpt.get("best_dice"),\n                          "init": ckpt.get("init", "pretrained")}\n        print(f"arm {name}: best_dice={refiners[name][\'best_dice\']}",\n              flush=True)\n\n    det_models = {n: YOLO(str(p)) for n, p in detectors}\n    started = time.time()\n\n    # 1) 全 detector の提案を det-floor で保持 (voter 入力は arm 共通)\n    props_by_file = {}\n    for name in oof_files:\n        image_path = args.preproc_dir / name\n        entries = []\n        for det_name, model in det_models.items():\n            result = model.predict(\n                str(image_path), imgsz=1280, conf=args.det_floor, iou=0.55,\n                agnostic_nms=True, retina_masks=True, verbose=False)[0]\n            masks, confidences = result_masks(result)\n            for m, s in zip(masks, confidences):\n                if s >= args.det_floor and m.sum() >= MIN_AREA:\n                    entries.append((det_name, m, float(s)))\n        props_by_file[name] = entries\n    print(f"proposals done: {time.time() - started:.0f}s", flush=True)\n\n    # 2) leader = yolo conf>=--conf のみ (検出集合固定)。voter は非 yolo 提案を\n    #    最良 IoU leader 1 つに割当 (mask-IoU >= --vote-iou)\n    def _mask_iou(a, b):\n        inter = int(np.logical_and(a, b).sum())\n        return inter / int(np.logical_or(a, b).sum()) if inter else 0.0\n\n    leaders_by_file = {}      # name -> [mask]\n    voters_by_file = {}       # name -> [[voter masks per leader idx]]\n    vote_diag = {"leaders_per_image": [], "voters_per_leader": [],\n                 "unassigned_voters": 0, "total_voters": 0}\n    for name in oof_files:\n        props = props_by_file[name]\n        leaders = [m for d, m, s in props\n                   if d == "yolo" and s >= args.conf]\n        candidates = [(d, m, s) for d, m, s in props if d != "yolo"]\n        per_leader = [[] for _ in leaders]\n        for d, m, s in candidates:\n            vote_diag["total_voters"] += 1\n            if not leaders:\n                vote_diag["unassigned_voters"] += 1\n                continue\n            ious = [_mask_iou(m, lm) for lm in leaders]\n            best = int(np.argmax(ious))\n            if ious[best] >= args.vote_iou:\n                per_leader[best].append(m)\n            else:\n                vote_diag["unassigned_voters"] += 1\n        leaders_by_file[name] = leaders\n        voters_by_file[name] = per_leader\n        vote_diag["leaders_per_image"].append(len(leaders))\n        vote_diag["voters_per_leader"].extend(len(v) for v in per_leader)\n    print(json.dumps({\n        "leaders_per_image": float(np.mean(vote_diag["leaders_per_image"]))\n            if vote_diag["leaders_per_image"] else 0.0,\n        "voters_per_leader": float(np.mean(vote_diag["voters_per_leader"]))\n            if vote_diag["voters_per_leader"] else 0.0,\n        "unassigned_voters": vote_diag["unassigned_voters"],\n        "total_voter_props": vote_diag["total_voters"]},\n        indent=1), flush=True)\n\n    # 3) leader + voter を同一 refiner で refine → arm 別 mask 合成 → PQ\n    ARM_NAMES = ("prod", "vote_eq05")\n    report = {"fold": args.fold, "folds_sha256": folds_sha256,\n              "provenance_manifest_sha256": args.provenance_sha256,\n              "detector_sha256": detector_shas,\n              "yolo_sha256": yolo_sha256,\n              "conf": args.conf, "det_floor": args.det_floor,\n              "vote_iou": args.vote_iou, "vote_rule": "equal-weight pixel majority >=0.5",\n              "min_area": MIN_AREA, "drop_threshold": DROP_THRESHOLD,\n              "thresholds": thresholds,\n              "leaders_per_image": float(np.mean(vote_diag["leaders_per_image"]))\n                  if vote_diag["leaders_per_image"] else 0.0,\n              "voters_per_leader": float(np.mean(vote_diag["voters_per_leader"]))\n                  if vote_diag["voters_per_leader"] else 0.0,\n              "unassigned_voters": vote_diag["unassigned_voters"],\n              "oof_files": len(oof_files), "arms": {}}\n    for arm_name, arm in refiners.items():\n        for prop_arm in ARM_NAMES:\n            meters = {t: PanopticQuality() for t in thresholds}\n            key = f"{arm_name}::{prop_arm}"\n            area_deltas = []\n            for name in oof_files:\n                image_path = args.preproc_dir / name\n                image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n                leaders = leaders_by_file[name]\n                voters = voters_by_file[name]\n                # leader/voter をまとめて refine (proposal 数を最小化)\n                all_props, index = [], []\n                for li, lm in enumerate(leaders):\n                    index.append((li, -1)); all_props.append(lm)\n                    for vi, vm in enumerate(voters[li]):\n                        index.append((li, vi)); all_props.append(vm)\n                prob_maps = [refine_probability(arm["model"], image, m)\n                             for m in all_props]\n                for threshold in thresholds:\n                    refined_idx = {}\n                    for (li, vi), (probability, box), proposal in zip(\n                            index, prob_maps, all_props):\n                        x0, y0, x1, y1 = box\n                        mask = np.zeros_like(proposal, dtype=bool)\n                        mask[y0:y1, x0:x1] = probability >= threshold\n                        refined_idx[(li, vi)] = mask & proposal\n                    predictions = []\n                    for li in range(len(leaders)):\n                        if prop_arm == "prod":\n                            predictions.append(refined_idx[(li, -1)])\n                            continue\n                        members = [refined_idx[(li, -1)]]\n                        members.extend(refined_idx[(li, vi)]\n                                       for vi in range(len(voters[li])))\n                        stack = np.stack(members).astype(np.float32)\n                        voted = stack.mean(axis=0) >= 0.5\n                        area_deltas.append(int(voted.sum())\n                                           - int(members[0].sum()))\n                        predictions.append(voted)\n                    predictions = apply_ownership(\n                        predictions, np.ones(len(predictions)),\n                        rule="drop_contained_then_area_asc",\n                        containment_threshold=DROP_THRESHOLD,\n                        min_area=MIN_AREA, canvas=EVAL_SIZE)\n                    pred_rles = (mask_utils.encode(np.asfortranarray(\n                        np.stack(predictions, axis=-1).astype(np.uint8)))\n                        if predictions else [])\n                    for image_id in file_to_ids[name]:\n                        truth = gt_rles[image_id]\n                        matrix = iou_matrix(truth, pred_rles, mask_utils)\n                        meters[threshold].add_matrix(matrix, len(truth),\n                                                     len(pred_rles))\n            report["arms"][key] = {\n                "refiner": arm_name, "proposal_arm": prop_arm,\n                "ckpt_sha256": arm["sha256"], "init": arm["init"],\n                "corpus_provenance": arm["corpus_provenance"],\n                "best_crop_dice": arm["best_dice"],\n                "emitted": int(np.mean([len(leaders_by_file[n])\n                                        for n in oof_files])),\n                "mean_area_delta_px": (float(np.mean(area_deltas))\n                                       if prop_arm == "vote_eq05" and area_deltas\n                                       else 0.0),\n                "by_threshold": {str(t): meters[t].summary()\n                                 for t in thresholds},\n            }\n            report["arms"][key]["pq_by_threshold"] = {\n                str(t): round(meters[t].score, 6) for t in thresholds}\n            print({key: report["arms"][key]["pq_by_threshold"]},\n                  flush=True)\n\n    report["minutes"] = round((time.time() - started) / 60, 1)\n    args.out.write_text(json.dumps(report, ensure_ascii=False, indent=2),\n                        encoding="utf-8")\n    print(json.dumps({a: r["pq_by_threshold"]\n                      for a, r in report["arms"].items()}, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
Path(code_dir / 'run_fold_oof_eval_vote.py').write_text(VOTE_SCRIPT)
import ast; ast.parse(VOTE_SCRIPT)
print('vote eval script written:', code_dir / 'run_fold_oof_eval_vote.py')
for name, p in [('preproc', preproc_dir), ('folds', folds_csv),
                ('baseline', baseline_ckpt), ('yolo', yolo_ckpt),
                ('crop', crop_ckpt), ('yolo26', yolo26_ckpt),
                ('train_json', train_json), ('code', code_dir)]:
    print(name, '->', p)
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
gpu_env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

MIRROR_DIR = Path('/content/drive/MyDrive/solar_filament_v125/work/latest')
def mirror():
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Drive not mounted')
    MIRROR_DIR.mkdir(parents=True, exist_ok=True)
    digests = {}
    for p in sorted(WORK.rglob('*')):
        if p.is_file() and 'input' not in p.relative_to(WORK).parts:
            dst = MIRROR_DIR / p.relative_to(WORK)
            expected = _sha(p)
            if dst.exists() and _sha(dst) == expected:
                digests[str(p.relative_to(WORK))] = expected; continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            if _sha(dst) != expected:
                raise RuntimeError(f'Drive mirror sha mismatch: {p.name}')
            digests[str(p.relative_to(WORK))] = expected
            print('mirrored', p.name, p.stat().st_size)
    (MIRROR_DIR / 'sha256_manifest.json').write_text(json.dumps(digests, indent=1))
    print('mirror DONE ->', MIRROR_DIR)


In [ ]:
%%time
# [eval] 2 arm — fold-0 OOF、prod leader 固定・cross-proposal 輪郭投票 (P2)
# crop/yolo26 det の byte-pin を実行時に照合 (trusted-sha256 受理の前提)
for path_str, expected in TRUSTED_SHA256.items():
    p = Path(path_str)
    actual = _sha(p)
    if actual != expected:
        raise RuntimeError(f'trusted sha256 mismatch: {p} {actual[:16]} != {expected[:16]}')
    print('trusted sha256 OK:', p, actual[:16])
trusted_args = [x for sha in TRUSTED_SHA256.values() for x in ('--trusted-sha256', sha)]
r = subprocess.run([sys.executable, str(code_dir / 'run_fold_oof_eval_vote.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '0',
                    '--provenance-manifest', str(manifest_path),
                    '--provenance-sha256', EXPECTED_PROVENANCE_SHA,
                    '--detector', f'yolo={yolo_ckpt}',
                    '--detector', f'crop={crop_ckpt}',
                    '--detector', f'yolo26={yolo26_ckpt}',
                    '--model', 'edgeatt',
                    *trusted_args,
                    '--refiner', f'baseline={baseline_ckpt}',
                    '--out', str(WORK / 'v125_eval.json')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-4000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'v125_eval.json').exists():
    raise RuntimeError(f'eval failed rc={r.returncode}')
mirror()


In [ ]:
# 結果サマリ — 開発側判定 (P2: vote_eq05 の fold-0 dev eval)
rep = json.load(open(WORK / 'v125_eval.json'))
arms = rep['arms']
pq = lambda a: arms[a]['pq_by_threshold']['0.45']
print(json.dumps({a: r['pq_by_threshold'] for a, r in arms.items()}, indent=2))
print('leaders/img:', rep.get('leaders_per_image'),
      ' voters/leader:', rep.get('voters_per_leader'),
      ' unassigned:', rep.get('unassigned_voters'))
base = pq('baseline::prod')
deltas = {a: round(pq(a) - base, 6) for a in arms if a != 'baseline::prod'}
rep_delta = deltas['baseline::vote_eq05']  # P2 の複製対象 arm
gate = {
  'prod_pq_0.45': base,
  'sanity_prod_near_v108': abs(base - 0.444137) < 0.001,
  'prod_diff_vs_v108': round(base - 0.444137, 6),
  'delta_by_arm': deltas,
  'vote_eq05_delta_fold0': rep_delta,
  'v121c_reference_delta': 0.002982,  # consensus 線 (閉鎖済み) 参照
  'note': 'fold-0 は v121c 判定済み — 陽性でも確認は別 fold が必要',
  'decision': ('vote_positive_dev' if rep_delta >= 0.005
               else 'vote_negative'),
}
print(json.dumps(gate, indent=2))
Path(WORK / 'v125_summary.json').write_text(json.dumps(
    {'eval': rep, 'gate': gate}, indent=2))
mirror()
